In [1]:
import re
import os
import sys
import math
import numpy as np
import pandas as pd

def get_genetic_code(file_path, choose):
    tl = open(file_path, "r")
    tl_content = tl.read()
    tl.close()
    tldict = {}
    read_active = False
    
    for line in tl_content.split("\n"):
        if not line.strip():
            continue
        if line.strip().split(" ")[0] == "Initiation" and read_active:
            tldict["alternative"] = line.strip().split(" ")[1:]
            read_active = False
        if read_active:
            line_list = line.strip().split(" ")
            tldict[line_list[0]] = line_list[1]
            tldict[line_list[8]] = line_list[9]
            tldict[line_list[16]] = line_list[17]
            tldict[line_list[24]] = line_list[25]
        if line.strip() == choose:
            read_active = True
    return tldict
        
def read_fasta(contents):
    content = open(contents, "r")
    align_content = content.read()
    sequence_name_before = "initialization"
    fasta = {}
    sequence = ''
    for line in align_content.split("\n"):
        if not line.strip():
            continue
        if line.startswith(">"):
            sequence_name_after = line.rstrip('\n').replace(">", "").split(" ", 1)[0]
            if sequence_name_after != sequence_name_before:
                fasta[sequence_name_before] = []
                fasta[sequence_name_before] = sequence
                sequence = ''
        else:
            sentence = line.rstrip('\n')
            sequence = sequence + sentence
            sequence_name_before = sequence_name_after
    fasta[sequence_name_before] = []
    fasta[sequence_name_before] = sequence
    del fasta['initialization']
    return fasta

def get_annotation(contents):
    annotatedtable = []
    prediction = open(contents, "r")
    prediction_content = prediction.read()
    for line in prediction_content.split("\n"):
        if not line.strip():
            continue
        if line[0] == "#":
            continue
        if line[0] == ">":
            break
        info = line.split("\t")
        Fragments = info[0]
        kind = info[2]
        start = int(info[3])
        stop = int(info[4])
        direction = info[6]
        annotation = info[8]
        ID = annotation.split("ID=")[1].split(";")[0]
        Uni = annotation.split("UniProtKB:")[1].split(";")[0] if 'UniProtKB' in annotation else None
        gene_name = annotation.split("Name=")[1].split(";")[0] if 'Name=' in annotation else None
        gene_product= annotation.split("product=")[1].split(";")[0] if 'product=' in annotation else None
        annotatedtable.append(pd.DataFrame({'ID':[ID],'fragments':[Fragments],'kind':[kind],'start':[start],'stop':[stop],
                                            'direction':[direction],'annotation':[annotation],
                                            'name':[gene_name],'product':[gene_product],'UniProt':[Uni]}))
    prediction.close()
    return pd.concat(annotatedtable, ignore_index=True)

def extract_dna(anno_tb, genome, tldict):
    Quick_Fragments = anno_tb.iloc[0,1]
    Quick_start = anno_tb.iloc[0,3] - 1
    Quick_stop = anno_tb.iloc[0,4]
    Quick_direction = anno_tb.iloc[0,5]
    if Quick_direction == "+":
        DNA = genome[Quick_start:Quick_stop]
    else:
        DNA = reverse_complement(genome[Quick_start:Quick_stop])
    if DNA[0:3] in tldict['alternative']:
        DNA = "ATG" + DNA[3:]
    RNA =""
    for i in range(0, len(DNA), 3): 
        codon = DNA[i:i + 3]
        codom_list = []
        for x in codon:
            codom_list.append(x)
        if len(set(codom_list) - set(["A","T","C","G"])) > 0:
            continue
        RNA = RNA + codon
    return DNA, RNA

def complement(seq):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A', 'N': 'N',
                  'R' : 'R','Y':'Y','W':'W','K':'K','M':'M','B':'B','D':'D',
                  'H':'H','V':'V'} 
    bases = list(seq) 
    bases = [complement[base] for base in bases] 
    return ''.join(bases)
def reverse_complement(s):
    return complement(s[::-1])
def translate(seq, tldict): 
    protein ="" 
    if len(seq)%3 == 0: 
        for i in range(0, len(seq), 3): 
            codon = seq[i:i + 3] 
            protein+= tldict[codon]
    else :
        seq = seq[0:(math.floor(len(seq)/3)*3)]
        for i in range(0, len(seq), 3): 
            codon = seq[tldict:i + 3] 
            protein+= tldict[codon]
    return protein

In [2]:
genome = read_fasta('ref/NC_000912.fasta')
genome = genome['NC_000912.1']

annotation = get_annotation('ref/NC_000912.gff3')
annotation = annotation[annotation['kind'].isin(['CDS','pseudogene'])]

# Get the genetic code from https://www.ncbi.nlm.nih.gov/Taxonomy/Utils/wprintgc.cgi
# The Mold, Protozoan, and Coelenterate Mitochondrial Code and the Mycoplasma/Spiroplasma Code (transl_table=4)
tl_path = "ref/Genetic_Codes.txt"
tldict = get_genetic_code(tl_path, '4')

In [3]:
dfs = pd.read_csv("subclade_pre.csv")
dfs = dfs[~dfs['site'].isna()]
dfs.reset_index(inplace=True, drop=True)
dfs['Parent'] = ''
dfs['product'] = ''
dfs['CDS_pos'] = ''
dfs['AA_pos'] = ''
dfs['mutation'] = ''
for i in range(dfs.shape[0]):
    sub_ann = annotation[(annotation['start'] < dfs.loc[i,'site']) & (annotation['stop'] > dfs.loc[i,'site'])]
    if sub_ann.shape[0] > 0:
        dfs.loc[i,'product'] = sub_ann['product'].values[0]
        if 'locus_tag' in sub_ann['annotation'].values[0]:
            word = sub_ann.annotation.values[0].split('locus_tag=')[1]
            dfs.loc[i,'Parent'] = word.split(';')[0]
        
        DNA, RNA = extract_dna(sub_ann, genome, tldict)
        if len(RNA)%3 != 0:
            continue
        aa = translate(RNA, tldict)

        dna_pos = dfs.loc[i,'site']-sub_ann.iloc[0,3] + 1
        aa_pos =  aa_pos = int(dna_pos/3) if dna_pos%3 == 0 else int(np.floor(dna_pos/3)+1)
        dfs.loc[i,'CDS_pos'] = dna_pos
        dfs.loc[i,'AA_pos'] = aa_pos

        ref_coden = DNA[((aa_pos-1)*3):(aa_pos*3)]
        ref_aa = aa[aa_pos-1]
        alt_site = dna_pos%3 - 1if dna_pos%3 != 0 else 2
        
        ALT_list = dfs.loc[i,'ALT'].split('&')
        ALT_list = [nt for nt in ALT_list if nt not in ['N']]
        mutation_type = []
        for alt_nt in ALT_list:
            if alt_nt == '-':
                mutation_type.append('Deletion')
                continue
            alt_coden = [alt_nt if ix == alt_site else nt  for ix, nt in enumerate(ref_coden)]
            alt_coden = ''.join(alt_coden)
            alt_aa = tldict[alt_coden]
            if alt_aa == ref_aa:
                mutation_type.append('Silent')
            elif alt_aa == '*':
                mutation_type.append('Nonsense')
            elif alt_aa != ref_aa:
                mutation_type.append('Missense')
        dfs.loc[i,'mutation'] = ','.join(mutation_type)

<>:29: SyntaxWarning: invalid decimal literal
<>:29: SyntaxWarning: invalid decimal literal
/tmp/ipykernel_2071486/2235000224.py:29: SyntaxWarning: invalid decimal literal
  alt_site = dna_pos%3 - 1if dna_pos%3 != 0 else 2


In [5]:
dfs

,site,REF,ALT,G_prop,nG_prop,p,padj,Parent,product,CDS_pos,AA_pos,mutation
0,59912,C,-&G,0.20,1.000000,4.636362e-09,1.159091e-06,MPN_RS00255,DUF240 domain-containing protein,294,98,"Deletion,Missense"
1,59914,T,-&C,0.20,1.000000,4.636362e-09,1.159091e-06,MPN_RS00255,DUF240 domain-containing protein,296,99,"Deletion,Missense"
2,116786,T,-,0.20,1.000000,4.636362e-09,1.159091e-06,,,,,
3,116787,C,-,0.20,1.000000,4.636362e-09,1.159091e-06,,,,,
4,128864,G,-,0.12,1.000000,2.059139e-10,5.147848e-08,MPN_RS04220,None,1470,490,Deletion
5,195459,C,-,1.00,0.033333,6.038185e-12,1.509546e-09,,,,,
6,340640,C,-,0.08,1.000000,3.766408e-11,9.416020e-09,MPN_RS01580,restriction endonuclease subunit S,397,133,Deletion
7,340649,A,-,0.08,1.000000,3.766408e-11,9.416020e-09,MPN_RS01580,restriction endonuclease subunit S,406,136,Deletion
8,340651,T,-,0.08,1.000000,3.766408e-11,9.416020e-09,MPN_RS01580,restriction endonuclease subunit S,408,136,Deletion
9,340652,C,-,0.08,1.000000,3.766408e-11,9.416020e-09,MPN_RS01580,restriction endonuclease subunit S,409,137,Deletion


In [6]:
# Variants with a prevalence lower than 95% in the target population were manually removed.
dfs.to_csv('TableS1.csv', index=False)

In [7]:
dfs = pd.read_csv("clade_specific_snp.csv")
dfs = dfs[~dfs['site'].isna()]
dfs.reset_index(inplace=True, drop=True)
dfs['Parent'] = ''
dfs['product'] = ''
dfs['CDS_pos'] = ''
dfs['AA_pos'] = ''
dfs['mutation'] = ''
for i in range(dfs.shape[0]):
    sub_ann = annotation[(annotation['start'] < dfs.loc[i,'site']) & (annotation['stop'] > dfs.loc[i,'site'])]
    if sub_ann.shape[0] > 0:
        dfs.loc[i,'product'] = sub_ann['product'].values[0]
        if 'locus_tag' in sub_ann['annotation'].values[0]:
            word = sub_ann.annotation.values[0].split('locus_tag=')[1]
            dfs.loc[i,'Parent'] = word.split(';')[0]
        
        DNA, RNA = extract_dna(sub_ann, genome, tldict)
        if len(RNA)%3 != 0:
            continue
        aa = translate(RNA, tldict)

        dna_pos = dfs.loc[i,'site']-sub_ann.iloc[0,3] + 1
        aa_pos =  aa_pos = int(dna_pos/3) if dna_pos%3 == 0 else int(np.floor(dna_pos/3)+1)
        dfs.loc[i,'CDS_pos'] = dna_pos
        dfs.loc[i,'AA_pos'] = aa_pos

        ref_coden = DNA[((aa_pos-1)*3):(aa_pos*3)]
        ref_aa = aa[aa_pos-1]
        alt_site = dna_pos%3 - 1if dna_pos%3 != 0 else 2
        
        ALT_list = dfs.loc[i,'ALT'].split('&')
        ALT_list = [nt for nt in ALT_list if nt not in ['N']]
        mutation_type = []
        for alt_nt in ALT_list:
            if alt_nt == '-':
                mutation_type.append('Deletion')
                continue
            alt_coden = [alt_nt if ix == alt_site else nt  for ix, nt in enumerate(ref_coden)]
            alt_coden = ''.join(alt_coden)
            alt_aa = tldict[alt_coden]
            if alt_aa == ref_aa:
                mutation_type.append('Silent')
            elif alt_aa == '*':
                mutation_type.append('Nonsense')
            elif alt_aa != ref_aa:
                mutation_type.append('Missense')
        dfs.loc[i,'mutation'] = ','.join(mutation_type)

<>:29: SyntaxWarning: invalid decimal literal
<>:29: SyntaxWarning: invalid decimal literal
/tmp/ipykernel_2071486/231683036.py:29: SyntaxWarning: invalid decimal literal
  alt_site = dna_pos%3 - 1if dna_pos%3 != 0 else 2


In [8]:
dfs

,clade,site,REF,ALT,clade_prop,others_prop,p,padj,Parent,product,CDS_pos,AA_pos,mutation
0,subclade_post_a,250217,T,-&C,0.750000,1.000000,6.305546e-16,1.222708e-11,MPN_RS04410,MPN207a family PTS transporter accessory protein,197,66,"Deletion,Missense"
1,subclade_post_a,250218,G,A&-,0.750000,1.000000,6.305546e-16,1.222708e-11,MPN_RS04410,MPN207a family PTS transporter accessory protein,198,66,"Silent,Deletion"
2,subclade_post_a,292495,C,A,0.000000,0.997863,1.755647e-82,3.404376e-78,MPN_RS01335,thioredoxin-disulfide reductase,168,56,Silent
3,subclade_post_a,377563,G,-&A&T,0.000000,0.993590,1.666742e-67,3.231979e-63,MPN_RS01790,APC family permease,356,119,"Deletion,Missense,Missense"
4,subclade_post_a,388615,C,T,0.000000,0.978632,2.913529e-41,5.649624e-37,MPN_RS01850,DUF3196 family protein,602,201,Missense
...,...,...,...,...,...,...,...,...,...,...,...,...,...
139,subclade_pre,638883,G,A,0.000000,1.000000,2.082172e-103,4.037540e-99,MPN_RS02970,MPN518 family protein,737,246,Silent
140,subclade_pre,685265,A,-,0.545455,0.071259,1.869598e-22,3.625337e-18,MPN_RS03235,NAD(P)-dependent alcohol dehydrogenase,537,179,Deletion
141,subclade_pre,738547,G,-,0.472727,0.061758,3.281688e-19,6.363522e-15,MPN_RS03500,restriction endonuclease subunit S,303,101,Deletion
142,subclade_pre,781943,C,T,0.836364,1.000000,4.052382e-15,7.857974e-11,MPN_RS03715,YitT family protein,65,22,Missense


In [9]:
# Variants with a prevalence lower than 95% in the target population were manually removed.
dfs.to_csv('TableS2.csv', index=False)